# Gradient descent from scratch

In [1]:
import numpy as np
import plotly.graph_objects as go
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

## Part 1: only b, on 4 points (slope fixed at the OLS value)

In [2]:
X, y = make_regression(n_samples=4, n_features=1, n_informative=1, n_targets=1, noise=80, random_state=13)
x = X.ravel()
ols = LinearRegression().fit(X, y)
print("OLS slope", ols.coef_[0], "intercept", ols.intercept_)

OLS slope 78.35063667947497 intercept 26.15963284313262


In [3]:
m, b, lr = 78.35, 100.0, 0.1
for step in range(1, 6):
    slope = -2 * np.sum(y - m * x - b)     # dL/db
    b = b - lr * slope                      # b_new = b_old - learning rate * slope
    print(f"step {step}: slope {slope:9.3f}  new b {b:8.3f}")

step 1: slope   590.722  new b   40.928
step 2: slope   118.144  new b   29.113
step 3: slope    23.629  new b   26.750
step 4: slope     4.726  new b   26.278
step 5: slope     0.945  new b   26.183


## Interactive: drag the slider to change the learning rate

In [4]:
bgrid = np.linspace(-150, 200, 300)
loss = lambda b: np.sum((y - 78.35 * x - b) ** 2)

def path(lr, steps=10, b=100.0):
    bs = [b]
    for _ in range(steps):
        bs.append(bs[-1] - lr * (-2 * np.sum(y - 78.35 * x - bs[-1])))
    return np.array(bs)

rates = [0.01, 0.03, 0.05, 0.1, 0.2, 0.24, 0.26]
fig = go.Figure([go.Scatter(x=bgrid, y=[loss(b) for b in bgrid], name="L(b)")] +
                [go.Scatter(x=path(r), y=[loss(b) for b in path(r)], mode="lines+markers", name=f"lr {r}",
                            visible=(r == 0.1)) for r in rates])
steps = [{"label": str(r), "method": "update",
          "args": [{"visible": [True] + [rr == r for rr in rates]}]} for r in rates]
fig.update_layout(sliders=[{"steps": steps, "active": 3, "currentvalue": {"prefix": "learning rate: "}}],
                  yaxis_range=[0, 100000], xaxis_title="b", yaxis_title="loss")
fig

## Part 2: m and b together, 100 points, our own class

In [5]:
X, y = make_regression(n_samples=100, n_features=1, n_informative=1, n_targets=1, noise=20, random_state=13)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
lr_ols = LinearRegression().fit(X_train, y_train)
print("OLS", lr_ols.coef_[0], lr_ols.intercept_, r2_score(y_test, lr_ols.predict(X_test)))

OLS 28.125973315135607 -2.271014426178382 0.6345158782661013


In [6]:
class GDRegressor:
    def __init__(self, learning_rate, epochs):
        self.m, self.b = 100.0, -120.0          # any starting values
        self.lr, self.epochs = learning_rate, epochs

    def fit(self, X, y):
        x = X.ravel()
        for _ in range(self.epochs):
            # slopes of the loss with respect to b and m
            slope_b = -2 * np.sum(y - self.m * x - self.b)
            slope_m = -2 * np.sum((y - self.m * x - self.b) * x)
            # update both together
            self.b -= self.lr * slope_b
            self.m -= self.lr * slope_m
        return self

    def predict(self, X):
        return self.m * X.ravel() + self.b

gd = GDRegressor(learning_rate=0.001, epochs=50).fit(X_train, y_train)
print("GD ", gd.m, gd.b, r2_score(y_test, gd.predict(X_test)))

GD  28.159367347119066 -2.3004574196824854 0.6343842836315579


In [7]:
# Test: update b first, then use the new b for the slope of m (sequential update)
x = X_train.ravel(); m, b = 100.0, -120.0
for _ in range(50):
    b -= 0.001 * (-2 * np.sum(y_train - m * x - b))
    m -= 0.001 * (-2 * np.sum((y_train - m * x - b) * x))
print("sequential", m, b)

sequential 28.157127055490708 -2.3000057908335005
